In [1]:
from langchain_gigachat.chat_models import GigaChat
import json
import pandas as pd
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.runnables import Runnable
from langchain.prompts import PromptTemplate
import re
from tqdm import tqdm
import json
from datetime import datetime
from itertools import islice

In [2]:
llm = GigaChat (
    #auth_url = "https://ngw.devices.sberbank.ru:9443/api/v2/oauth",
    scope = "GIGACHAT_API_CORP",
    credentials="GIGACHAT_CREDENTIALS_REMOVED",
    verify_ssl_certs=False,
    model = "GigaChat-2-Max",
    timeout=300
)

In [4]:
weekday_translation = {
    0: "Понедельник",
    1: "Вторник",
    2: "Среда",
    3: "Четверг",
    4: "Пятница",
    5: "Суббота",
    6: "Воскресенье"
}

with open('mp.json', 'r', encoding='utf-8') as file:
    data = json.load(file)

for event in data:
    date_str = event["date"]
    date_obj = datetime.strptime(date_str, "%Y-%m-%d")
    weekday_num = date_obj.weekday()
    event["weekday"] = weekday_translation[weekday_num]

with open('mp.json', 'w', encoding='utf-8') as file:
    json.dump(data, file, ensure_ascii=False, indent=2)

In [5]:
parser = JsonOutputParser()

In [6]:
with open("analysis_last.json", "r", encoding="utf-8") as f:
    json1 = json.load(f)

with open("mp.json", "r", encoding="utf-8") as f:
    json2 = json.load(f)

In [7]:
json1_str = json.dumps(json1, ensure_ascii=False, indent=2)
json2_str = json.dumps(json2, ensure_ascii=False, indent=2)

In [15]:
prompt_template3 = PromptTemplate(
    input_variables=["json1", "json2"],
    template="""
[СИСТЕМНАЯ ИНСТРУКЦИЯ]
Ты — специалист по распределению клиентов по мероприятиям исходя из входных данных. 
Твоя задача — сопоставить как можно больше клиентов с мероприятиями на основе их тегов.  

[АЛГОРИТМ РАБОТЫ]
1. Проанализируй мероприятие {json2}, его название и его теги.
2. Проанализируй каждого клиента {json1} и его теги.
3. Проверь, подходят ли теги клиента для мероприятия.
    Тег может подходить под навазние мероприятия и под формат мероприятия по смыслу или может быть связан с одной тематикой.
    Например на мероприятие "Детский праздник 'Отцы и дети'" добавляй клиентов с тегами "Культурный досуг"
4. Если теги подходят — добавь клиента в список мероприятия.
5. Повтори шаги 1-5 для каждого мероприятия.

[ОСОБЕННОСТИ РАБОТЫ]
1. На мероприятие может быть приглашено несколько клиентов.
2. Если теги клиента не подходят для мероприятия — не приглашаем клиента.
3. Если тег клиента подходит по смыслу к мероприятию - добавь его в список.

[ПРАВИЛА]
Обязательно:  
- Используй валидный JSON (двойные кавычки ", запятые между элементами).  
- Сохраняй исходные названия тегов (без изменений).  

Запрещено:  
- Добавлять текстовые пояснения вне JSON.  
- Менять структуру входных данных. 

[
  {{
    "title": "...",
    "event_tags": ["...", "...", "..."],
    "date": "....-..-..",
    "time": "..:..",
    "weekday": "...",
    "prof": true or false,
    "matched_people": [
      {{
        "FIO": "...",
        "matching_tags": [
          "...",
          "..."
        ]
      }},
      {{
        "FIO": "...",
        "matching_tags": [
          "...",
          "..."
        ]
      }}
    ]
  }}
]
"""
)

In [16]:
chain4: Runnable = prompt_template3 | llm | parser

In [17]:
fin = chain4.invoke({"json1": json1, "json2": json2})

In [18]:
with open("fin_matches.json", "w", encoding="utf-8") as f:
    json.dump(fin, f, ensure_ascii=False, indent=2)